# Practicing Four LLM Prompting Techniques
**Workshop:** Dr. Bouy Rina · Techpreneur 2.0  
**Student:** SAN Piseth

This worked notebook completes the four required exercises from the instructor's prompting notebook. It includes code, explanations, result checks, and questions to guide your interpretation.

**Execution status:** Model calls have not been run in the preparation environment. Run this notebook in Colab to produce your own outputs. Expected answers below are reference answers, not claimed model results.

**Source:** [Dr. Rina's original notebook](https://github.com/rinabuoy/llm-fundamentals-tp2/blob/main/notebooks%20-%20session%202/LLM_Prompting_Ollama_Colab_TP.ipynb). The setup and demonstrations below are adapted from the provided file. The homework solutions and explanations have been added.

## Prepare Colab
Upload this notebook to Google Colab, select a GPU runtime, and run cells in order. The setup cell installs Ollama and downloads the model specified in the instructor's notebook. It is intended for Colab's Linux environment, not directly for Windows. Model availability and GPU capacity can affect execution. If setup fails, stop and inspect the error before continuing.

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [2]:
import subprocess, time

%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
ollama_log = open("ollama.log", "w")
ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
)
time.sleep(5); time.sleep(5)
!ollama pull gemma4:e2b

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
The system cannot find the path specified.
The system cannot find the path specified.
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling 

In [3]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [4]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [5]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
$50 - 18 = 32$ shirts remaining.

**3. Shipment Received:**
They received a shipment of 30 shirts.
$32 + 30 = 62$ shirts total.

**4. Second Sale:**
They sold half of what they had (half of 62).
Half of 62 is $62 / 2 = 31$ shirts sold.

**5. Final Count:**
Subtract the second sale from the current total:
$62 - 31 = 31$ shirts left.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [6]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: I do not have access to specific return policies for Acme Corp. To find out how many days you have to return an item, you will need to check their official website, return policy page, or contact Acme Corp directly.
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


## Exercise 1: Extract information into JSON
**Goal:** Compare zero-shot prompting with exactly two examples.

1. Keep the same target sentence and extraction instruction for both prompts.
2. Run the zero-shot prompt without examples.
3. Add two examples with different names, cities, and ages.
4. Inspect both raw responses and check whether each is valid JSON with the correct values.

The reference answer is `{"name": "Sokha", "city": "Siem Reap", "age": 29}`. Examples show the required format; they do not update the model's weights.

In [8]:
import json

text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."
instruction = 'Extract name, city, and age as JSON. Use keys "name", "city", and "age". Use an integer for age. Return only the JSON object.'

zero_prompt = f"{instruction}\n\nText: {text}\nJSON:"
few_prompt = instruction + """

Text: Dara is 22 years old and lives in Phnom Penh.
JSON: {"name": "Dara", "city": "Phnom Penh", "age": 22}

Text: Lina, aged 35, lives in Battambang.
JSON: {"name": "Lina", "city": "Battambang", "age": 35}

""" + f"Text: {text}\nJSON:"

expected = {"name": "Sokha", "city": "Siem Reap", "age": 29}
ex1_results = {}
for label, prompt in [("Zero-shot", zero_prompt), ("Two-example", few_prompt)]:
    output = generate(prompt, max_tokens=120, temperature=0.0)
    print(f"\n{label} raw output:\n{output}")
    try:
        parsed = json.loads(output)  # Strict: prose or Markdown fences are not raw JSON.
        correct = (isinstance(parsed, dict) and parsed == expected
                   and type(parsed.get("age")) is int)
        print("Valid JSON: Yes | Correct schema and values:", correct)
    except json.JSONDecodeError:
        correct = False
        print("Valid JSON: No — inspect the raw response above.")
    ex1_results[label] = {"output": output, "correct": correct}
print("\nComparison:", {k: v["correct"] for k, v in ex1_results.items()})


Zero-shot raw output:
```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```
Valid JSON: No — inspect the raw response above.

Two-example raw output:
{"name": "Sokha", "city": "Siem Reap", "age": 29}
Valid JSON: Yes | Correct schema and values: True

Comparison: {'Zero-shot': False, 'Two-example': True}


**Interpretation:** Few-shot examples can help the model follow a format. Both prompts may already succeed on this simple sentence; that is a valid result. If both checks pass, say that adding examples did not improve this particular test. If only the two-example response passes, describe the specific formatting or extraction error it fixed. One sentence is not enough to establish a general performance advantage.

**After running:** Write 2–3 sentences here describing your actual outputs.

## Exercise 2: Compare direct and step-by-step answers
**Goal:** Test the bat-and-ball puzzle using two prompt styles.

The correct ball price is **$0.05**: if the ball costs x dollars, the bat costs x + 1.00, so 2x + 1.00 = 1.10. Therefore x = 0.05. A $0.10 ball would make the total $1.20, not $1.10.

Both prompts specify dollars and the same final-answer format. The second also requests a brief step-by-step solution.

In [9]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"
direct = f"{puzzle}\nGive only the final price in dollars, in the format Answer: <number>."
cot = f"{puzzle}\nSolve step by step briefly, then end with the price in dollars as Answer: <number>."

direct_output = generate(direct, max_tokens=60, temperature=0.0)
cot_output = generate(cot, max_tokens=300, temperature=0.0)
print("Direct:\n", direct_output)
print("\nStep-by-step:\n", cot_output)
print("\nReference answer: $0.05")

Direct:
 Let $B$ be the cost of the bat and $L$ be the cost of the ball.

We are given two pieces of information:
1. The total cost of the bat and the ball is $1.10:
   $$B + L = 1.1

Step-by-step:
 Here is the step-by-step solution:

**1. Define variables:**
Let $B$ be the cost of the bat.
Let $L$ be the cost of the ball.

**2. Set up equations based on the given information:**
Total cost: $B + L = 1.10$ (Equation 1)
Cost relationship: The bat costs $1.00 more than the ball, so $B = L + 1.00$ (Equation 2)

**3. Solve the system of equations:**
Substitute Equation 2 into Equation 1:
$(L + 1.00) + L = 1.10$

**4. Solve for L (the cost of the ball):**
Combine like terms:
$2L + 1.00 = 1.10$
Subtract $1.00$ from both sides:
$2L = 1.10 - 1.00$
$2L = 0.10$
Divide by 2:
$L = 0.10 / 2$
$L = 0.05$

**5. Check the answer (Optional):**
If the ball ($L$) costs $\$0.05$, the bat ($B$) costs $\$0.05 + \$1.00 = \$1.0

Reference answer: $0.05


**Interpretation:** Check each answer against $0.05. Also check whether the displayed steps correctly express the price difference. A longer answer is not automatically correct. If both prompts give $0.05, both succeeded; do not claim the direct prompt failed.

**After running:** Record each answer, which is correct, and whether the step-by-step explanation helped you verify it.

## Exercise 3: Choose the most common answer
**Goal:** Implement `self_consistent(prompt, k)`, sample at **temperature=0.8**, and compare **k=1** with **k=7**.

For each response, extract the final `Answer:` value. Normalize equivalent dollar values so `0.05` and `0.050` count together. Do not count missing answers as votes. Print every response and the vote counts so the result is reviewable. A tie produces no unique winner. The most common answer is a plurality; the code also reports whether it has a strict majority of valid votes.

Using seven responses costs more generation time. Voting may help, but repeated responses can share the same mistake.

In [10]:
from decimal import Decimal, InvalidOperation
from collections import Counter
import re

def extract_dollar_answer(text):
    matches = re.findall(r"Answer\s*:\s*\$?\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*(cents?|dollars?|USD)?", text, flags=re.I)
    if not matches:
        return None
    number, unit = matches[-1]
    try:
        value = Decimal(number)
        if unit.lower().startswith("cent"):
            value /= 100
        return format(value.normalize(), "f")
    except InvalidOperation:
        return None

def self_consistent(prompt, k=5):
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError("k must be a positive integer")
    answers = []
    for i in range(k):
        output = generate(prompt, max_tokens=350, temperature=0.8)
        answer = extract_dollar_answer(output)
        print(f"\nSample {i + 1}/{k}:\n{output}\nParsed dollars: {answer}")
        answers.append(answer)
    counts = Counter(a for a in answers if a is not None)
    print("\nVotes:", dict(counts), "| Unparsed:", answers.count(None))
    if not counts:
        print("No valid votes; inspect the responses and prompt format.")
        return None
    highest = max(counts.values())
    winners = [a for a, count in counts.items() if count == highest]
    if len(winners) != 1:
        print("Tie: no unique winner.")
        return None
    winner = winners[0]
    print("Winner:", winner, "| Strict majority of valid votes:", highest > sum(counts.values()) / 2)
    return winner

cot_prompt = cot
print("=== k=1 ===")
winner_1 = self_consistent(cot_prompt, k=1)
print("\n=== k=7 ===")
winner_7 = self_consistent(cot_prompt, k=7)
print("\nComparison with reference $0.05:")
for k, winner in [(1, winner_1), (7, winner_7)]:
    print(f"k={k}: winner={winner}, correct={winner == '0.05'}")

=== k=1 ===

Sample 1/1:
Here is the step-by-step solution:

**1. Define Variables:**
Let $B$ be the cost of the bat.
Let $L$ be the cost of the ball.

**2. Set Up Equations based on the problem:**
The total cost is $1.10:
$$B + L = 1.10 \quad \text{(Equation 1)}$$
The bat costs $\$1.00$ more than the ball:
$$B = L + 1.00 \quad \text{(Equation 2)}$$

**3. Solve the System of Equations:**
Substitute Equation 2 into Equation 1:
$$(L + 1.00) + L = 1.10$$

**4. Solve for L (the cost of the ball):**
Combine the terms with $L$:
$$2L + 1.00 = 1.10$$
Subtract $1.00$ from both sides:
$$2L = 1.10 - 1.00$$
$$2L = 0.10$$
Divide by 2:
$$L = 0.10 / 2$$
$$L = 0.05$$

**5. Verification (Optional):**
If the ball ($L$) costs $\$0.05$, the bat ($B$) costs $\$0.05 + \$1.00 = \$1.05$.
Total cost: $\$1.05 + \$0.05 = \$1.10$. (This matches the problem.)

The price of the
Parsed dollars: None

Votes: {} | Unparsed: 1
No valid votes; inspect the responses and prompt format.

=== k=7 ===

Sample 1/7:
Here is th

**Interpretation:** If k=1 is wrong and k=7 is correct, voting helped in this run. If both are correct, there is no observed accuracy improvement on this puzzle. If the seven-sample winner is wrong, voting repeated or amplified a shared error. If parsing fails, inspect the raw output rather than treating it as an incorrect numerical answer. This small comparison cannot establish a reliable overall success rate.

**After running:** Note the two winners, the seven-sample vote counts, and whether voting helped.

## Exercise 4: Answer using retrieved facts
**Goal:** Add two facts, rebuild the TF-IDF index, and compare answers with and without retrieval.

The new facts below are **fictional exercise data** about Acme Corp. TF-IDF represents words numerically; cosine similarity ranks documents by overlap with the query. Re-fitting is necessary because the collection now contains new words and documents.

We test both new facts and an unsupported question about the CEO. The unknown question deliberately shares the company name with the documents: retrieving a related document does not mean it contains an answer.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

base_docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
new_facts = [
    "Acme Corp's student discount is 17 percent with a valid student ID.",
    "Acme Corp's delivery fee within Siem Reap is $2.75 per order.",
]
docs = base_docs + new_facts  # Re-running does not duplicate facts.
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=2):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    indices = scores.argsort()[::-1][:k]
    return [docs[i] for i in indices]

def rag_answer(query, k=2):
    context = "\n".join(retrieve(query, k))
    prompt = f"""Answer the question using only the context below.
Do not add outside knowledge or guess missing facts.
If the context does not contain the answer, reply exactly: I don't know.
Keep supported answers brief.

Context:
{context}

Question: {query}
Answer:"""
    return generate(prompt, max_tokens=120, temperature=0.0)

questions = [
    "What is Acme Corp's student discount?",
    "What is Acme Corp's delivery fee within Siem Reap?",
    "Who is the CEO of Acme Corp?",
]
ex4_results = []
for q in questions:
    without = generate(f"Question: {q}\nAnswer:", max_tokens=120, temperature=0.0)
    with_rag = rag_answer(q)
    ex4_results.append({"question": q, "without": without, "with_rag": with_rag})
    print("\nQuestion:", q)
    print("Retrieved context:", retrieve(q))
    print("Without retrieval:", without)
    print("With retrieval:", with_rag)
print("\nReference checks: 17 percent; $2.75 per order; I don't know.")


Question: What is Acme Corp's student discount?
Retrieved context: ["Acme Corp's student discount is 17 percent with a valid student ID.", "Acme Corp's delivery fee within Siem Reap is $2.75 per order."]
Without retrieval: Please provide me with some context or a document. I do not have access to specific, real-time, or private company information like Acme Corp's student discount unless you provide that information to me.
With retrieval: Acme Corp's student discount is 17 percent with a valid student ID.

Question: What is Acme Corp's delivery fee within Siem Reap?
Retrieved context: ["Acme Corp's delivery fee within Siem Reap is $2.75 per order.", "Acme Corp's student discount is 17 percent with a valid student ID."]
Without retrieval: Please provide the document or context you are referring to so I can tell you Acme Corp's delivery fee within Siem Reap.
With retrieval: $2.75 per order.

Question: Who is the CEO of Acme Corp?
Retrieved context: ["Acme Corp's delivery fee within Siem

**Interpretation:** Compare answers with the fictional source facts: 17 percent and $2.75 per order. Without retrieval, the model might guess or decline; record what it actually does. With retrieval, the prompt supplies evidence for the answer. For the CEO question, none of the documents gives a name, so the desired answer is “I don't know.” An instruction reduces guessing but does not guarantee it; if the model invents a name, report that failure and test a clearer prompt.

**After running:** Describe the supported answers, the unsupported answer, and whether the model followed the instruction.

## Finish and submit
1. Confirm your name at the top: SAN Piseth.
2. Run all cells in order and resolve execution errors. Setup and the first model call may take time.
3. Write your observations in the four interpretation sections using the displayed outputs.
4. Save/download the notebook **with outputs**. Do not present the reference answers as actual model responses.
5. Upload the notebook to your own GitHub repository or fork. Add a short README describing the model, runtime, exercises, and any limitations.
6. In a plain-text file such as `submission.txt`, include your name, repository URL, and direct notebook URL. Replace the placeholders below with your actual links, then attach that file to the assignment submission page.

```text
Name: SAN Piseth
Assignment: Practicing Four LLM Prompting Techniques
Repository: YOUR GITHUB REPOSITORY URL
Notebook: DIRECT GITHUB NOTEBOOK URL
```

The optional challenge and OCR fine-tuning demo are not included in this four-exercise solution.

**Takeaways:** Examples can guide output formatting; step-by-step prompts can make a solution easier to verify; voting combines multiple sampled answers; retrieval provides task-specific evidence. None of these techniques guarantees correctness.